# Reto 01 · Hito 3: agrupamiento, PCA y perfiles municipales

**Aprendizaje Automático Inicial · Ciencia de Datos, PUCE · Periodo 2026-2**  
**Pregunta esencial (Reto01.pdf):** ¿Qué perfiles de gestión de residuos aparecen en los registros municipales del Ecuador y cómo comunicar sus diferencias para orientar preguntas y propuestas de mejora?

Este notebook es un **borrador técnico del Hito 3**, preparado para la revisión del equipo y el posterior Hito 4. Trabaja exclusivamente con las matrices obtenidas en el Hito 2 desde GIRS **2024** del INEC (no otras ediciones). No produce una clasificación oficial ni un ranking municipal.

**Mapa de actividades guía:** 5. K-Means y Ward; 6. silueta, tamaños, semillas y estabilidad; 7. PCA; 8. perfiles, ejemplos y oportunidades de exploración. Las actividades 1–4 están documentadas en los hitos anteriores. 

## 0. Fuente y alcance

Fuente institucional: [INEC / ANDA, GIRS 2024](https://anda.inec.gob.ec/anda5/index.php/catalog/1294/data-dictionary); [documentos y formulario](https://anda.inec.gob.ec/anda5/index.php/catalog/1294/related-materials). El archivo oficial completo posee **221 municipios y 213 columnas**. Este notebook utiliza el subconjunto de **41 columnas relevantes exportadas** y las matrices ya preparadas del equipo, no suplanta al CSV original.

Hito 2: **215** municipios como escenario principal y **204** como sensibilidad. Cinco valores de cobertura de barrido sin denominador y un cociente diferenciado inválido impiden casos completos; once ceros de recuperación cuestionables solo se excluyen en el segundo escenario. Se conservan coberturas `CSB > 1` y los municipios con perfiles numéricos iguales. No se utilizan identificadores en distancias. 

## 1. Preparar el entorno de Colab

La siguiente celda clona el repositorio público o lo actualiza. **No necesitas subir manualmente los CSV**, pero debes tener conexión y acceso a GitHub. Si el repositorio cambia, la ejecución de Colab tomará la versión vigente; los resultados publicados se obtuvieron con el commit que consta en GitHub.


In [ ]:
import os, sys, subprocess, csv, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

local = os.environ.get('GIRS_REPO_ROOT_LOCAL', '')
if local and Path(local).exists():
    ROOT = Path(local).resolve()  # Para verificación local sin conexión
else:
    ROOT = Path('/content/Aprendizaje_Automatico_Reto01')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/2007Mateo/Aprendizaje_Automatico_Reto01.git', str(ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
print('Repositorio:',ROOT)
assert (ROOT/'02_hito2/matrices/matriz_215.csv').exists(), 'Falta matriz Hito 2'
assert (ROOT/'03_hito3/codigo/ejecutar_hito3.py').exists(), 'Falta código Hito 3: actualiza GitHub'


## 2. Auditar la matriz recibida (actividades 1–4)

La matriz usa las cinco características justificadas en Hito 2: cobertura de barrido (proporción), proporción de recolección diferenciada, indicador binario de actividad orgánica, recuperación inorgánica relativa transformada con logaritmo y botadero como disposición principal. Se guarda `CantonId` exclusivamente para vincular los resultados **después** de calcular grupos.


In [ ]:
def leer_csv(path):
    with open(path, encoding='utf-8-sig', newline='') as archivo:
        return list(csv.DictReader(archivo))

cols = ['CSB_cobertura','DIF_proporcion','ORG_actividad','RI_log_p95','BOTADERO_indicador']
rows215=leer_csv(ROOT/'02_hito2/matrices/matriz_215.csv')
rows204=leer_csv(ROOT/'02_hito2/matrices/matriz_204.csv')
base221=leer_csv(ROOT/'02_hito2/datos/base_preparada_221.csv')
incidencias=leer_csv(ROOT/'02_hito2/datos/incidencias_17.csv')
X215=np.array([[float(z[c]) for c in cols] for z in rows215])
X204=np.array([[float(z[c]) for c in cols] for z in rows204])
assert (len(base221), X215.shape, X204.shape, len(incidencias)) == (221, (215,5), (204,5),17)
assert np.isfinite(X215).all() and np.isfinite(X204).all()
print('Edición: INEC GIRS 2024; matrices 215x5 / 204x5; 221 registros auditados, 17 incidencias')
for i,c in enumerate(cols):
    print(f'{c:22s} min={X215[:,i].min():.3f}  mediana={np.median(X215[:,i]):.3f}  máx={X215[:,i].max():.3f}')
print('Perfiles numéricos únicos:',len(np.unique(X215,axis=0)),'de',len(X215))


## 3. Actividades 5 y 6: comparar K-Means y jerárquico Ward

Configuraciones: `k = 2,3,4,5,6`; Ward con enlace Ward y distancia euclidiana; K-Means con 5 semillas (`11,23,37,52,97`) y `n_init=30`, para cubrir variación entre inicializaciones. La silueta se calcula **en las cinco dimensiones utilizadas para agrupar**, no en PCA 2D. La concordancia de etiquetas se mide con ARI (invariante a su numeración).


In [ ]:
import subprocess
run = subprocess.run([sys.executable, str(ROOT/'03_hito3/codigo/ejecutar_hito3.py')],cwd=ROOT,text=True,capture_output=True)
print(run.stdout)
if run.returncode:
    print(run.stderr)
    raise RuntimeError('El script no terminó correctamente')
OUT=ROOT/'03_hito3/resultados'
FIG=ROOT/'03_hito3/figuras'


In [ ]:
comp=leer_csv(OUT/'comparacion_kmeans_ward_204_215.csv')
for k in range(2,7):
    a=next(x for x in comp if x['escenario']=='215' and x['algoritmo']=='K-Means' and int(x['k'])==k)
    b=next(x for x in comp if x['escenario']=='215' and x['algoritmo']=='Ward' and int(x['k'])==k)
    print(f'k={k} | K-Means={float(a["silueta_5d"]):.3f} | Ward={float(b["silueta_5d"]):.3f} | tamaños KMeans={a["tamanos_por_cluster"]} | ARI entre métodos={float(a["ari_kmeans_vs_ward"]):.3f}')
display(Image(filename=str(FIG/'01_comparacion_silueta.png')))
display(Image(filename=str(FIG/'02_tamanos_k5.png')))


## 4. Elección responsable: k=5 como descripción principal, k=4 y k=6 como contrastes

Se usa **K-Means, k=5, semilla 11**, como solución **descriptiva provisional**. No es automáticamente la mejor: `k=6` muestra silueta superior, pero forma un grupo pequeño de alta recuperación, potencialmente sensible a valores extremos; `k=4` es más compacto y muy estable al retirar los once ceros ambiguos. En `k=5` hay cinco patrones sustantivos, grupos con al menos 18 municipios, y concordancia entre algoritmos. La decisión debe explicarse por estabilidad, tamaño e interpretación, y revisarse en el informe final.


In [ ]:
print('Estabilidad al excluir 11 ceros, sobre los 204 municipios compartidos:')
for r in leer_csv(OUT/'sensibilidad_204_vs_215.csv'):
    if r['algoritmo']=='K-Means':
        print('k',r['k'],'ARI:',r['ARI_215_contra_204'],'cambios de asignación tras permutar etiquetas:',r['cambios_optimos_municipios'])
print('\nCambios al escalar/reemplazar/añadir cantidades:')
for r in leer_csv(OUT/'sensibilidad_escalado_absolutos_volumen.csv'):
    if r['algoritmo']=='K-Means': print(r['experimento'], '| ARI',r['ARI_vs_matriz_principal'])
display(Image(filename=str(FIG/'08_sensibilidad_204_vs_215.png')))


## 5. Actividad 7: PCA para visualización, no evaluación

Los dos componentes permiten ubicar municipios en una figura. Se informa **varianza explicada de cada eje y acumulada**, y **contribución porcentual de cada variable** calculada como el cuadrado del peso del autovector normalizado. Los signos pueden cambiar entre ejecuciones sin modificar la interpretación sustantiva.


In [ ]:
info=json.loads((OUT/'diagnostico_escala_y_modelo.json').read_text(encoding='utf-8'))
print('Varianza explicada PC1 y PC2:',*[f'{x:.2f}%' for x in info['pca_varianza_pct'][:2]])
print('Varianza acumulada PC1+PC2:',f'{sum(info["pca_varianza_pct"][:2]):.2f}%')
pca=leer_csv(OUT/'pca_varianza_pesos_contribuciones.csv')
for pc in ['1','2']:
    print('Contribuciones principales PC'+pc, sorted([(r['caracteristica'],float(r['contribucion_eje_pct'])) for r in pca if r['componente']==pc],key=lambda x:-x[1])[:3])
display(Image(filename=str(FIG/'03_pca_kmeans_k5.png')))
display(Image(filename=str(FIG/'04_pca_ward_k5.png')))
display(Image(filename=str(FIG/'05_pca_contribuciones.png')))


## 6. Actividad 8: perfiles, ejemplos y límites

La tabla se construye sobre **unidades originales**: porcentajes de cobertura y diferenciación, porcentaje de municipios con actividad orgánica y botadero, mediana de recuperación expresada en kg/mes y mediana de recolección en toneladas/mes. Los números de grupo **no son calificaciones**. Los ejemplos se seleccionaron por cercanía al centroide en las cinco dimensiones, no por prestigio.


In [ ]:
perfiles=leer_csv(OUT/'perfiles_kmeans_k5_215.csv')
for p in perfiles:
    print(f'G{p["id_grupo"]} ({p["municipios"]}): {p["perfil"]}; diferenciada media={p["diferenciada_media_pct"]}%; orgánica={p["actividad_organica_pct_municipios"]}%; botadero={p["botadero_pct_municipios"]}%; RI mediana={p["RI_kg_mes_mediana"]} kg/mes')
print('\nEjemplos representativos:')
for p in leer_csv(OUT/'municipios_representativos_k5.csv'):
    if p['representatividad']=='1':print('G'+p['id_grupo'], p['NombreCantón'],'(código publicado:',p['CantonId']+')')
display(Image(filename=str(FIG/'06_perfiles_porcentajes.png')))
display(Image(filename=str(FIG/'07_silueta_individual.png')))


### Interpretación crítica y necesidades de información

- Grupos con recolección diferenciada o actividad orgánica declarada **no equivalen** a mayor recuperación efectiva: las respuestas binarias describen actividades, no rendimientos.
- El grupo con botadero principal no es un ranking de calidad; conviene estudiar condiciones técnicas, infraestructura y controles, con documentación adicional.
- Se observaron **11 ceros de RI ambiguos**: no se infieren cantidades ni se eliminan sin examinar la sensibilidad 204/215. Asimismo, `CSB > 1` se preservó sin recortes.
- `CantonId` procede de archivos del equipo; parte de sus ceros iniciales pudo perderse en Excel. No reconstruir identificadores sin el CSV oficial.
- La pregunta sobre escala importa: al incluir `log1p(CRR)` como característica, la partición se modifica de forma importante. El clustering principal usa indicadores de prácticas y la cantidad recolectada solo como contexto.
- Se requieren visitas, documentación municipal, infraestructura de recolección, series temporales, validación administrativa y detalles de tratamiento antes de sugerir decisiones de inversión o política.

**Fuentes:** INEC, edición GIRS 2024; Reto01.pdf (Andrés Merino); documentación scikit-learn de [K-Means / Ward](https://scikit-learn.org/stable/modules/clustering.html), [silueta](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html), [ARI](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html) y [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html). El archivo `documentacion/referencias.bib` registra citas para la entrega final.


## 7. Conclusión técnica del Hito 3

Los registros municipales 2024 muestran **perfiles exploratorios diferentes** en separación, actividad orgánica y disposición en botadero. Cinco grupos son útiles como borrador interpretativo, pero la estabilidad se ha evaluado con varias semillas, otro algoritmo, dos escenarios de limpieza y transformaciones de escala. PCA es una ayuda gráfica, no una certificación de separación. No usar las etiquetas para ordenar municipios ni distribuir recursos automáticamente.

Este notebook, junto con las tablas y figuras, constituye el material de **Hito 3** a entregar a la persona encargada de **Hito 4**. No reemplaza el informe divulgativo final en el formato LaTeX PUCE.


## 8. Opcional: descargar todo el paquete de resultados desde Colab

Descomenta la descarga si quieres una copia local (en Colab, los archivos de `/content` se pierden al cerrar el entorno). También están versionados en GitHub.


In [ ]:
import shutil
zip_path=shutil.make_archive(str(ROOT/'Hito3_resultados_2024'),'zip',root_dir=ROOT/'03_hito3')
print('ZIP local:',zip_path)
# En Colab, descomenta para enviarlo a tu computadora:
# from google.colab import files
# files.download(zip_path)
